# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [1]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append("/home/shikim/pynta")

from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface
from ase.visualize import view

import site_analysis as sa


In [2]:
# USER INPUT
name = '/home/shikim/pynta/pynta/preprocessing/custom_surfaces/pd332_surface/pd332.xyz'
n_layers = 10
#atoms = surface('Pd',(3,3,2),n_layers) #lattice constant
#atoms.center(vacuum=10, axis=2)


In [3]:
#visualize slab
write(name, atoms)
view(atoms, viewer='x3d')

NameError: name 'atoms' is not defined

In [3]:

# Load slab
slab = read(name)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [ ]:
# rattled surface
slabrat = slab.copy()
slabrat.rattle(stdev=0.3)
view(slabrat, viewer='x3d')

# do not rattle the surface!!

In [4]:
adsorbate_height = 2
site_bond_cutoff = 2

slabrat = slab.copy()
#slabrat.rattle(stdev=0.3)

# Generate symmetry-unique site geometries
cas = SlabAdsorptionSites(slab, "fcc332", composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slab, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())

#traj = Trajectory("unique_sites.traj", "w")
#for g in single_geoms:
#    traj.write(g)
#traj.close()

#do not rattle the surface. it distorts the surface

# ... your code that makes single_sites_lists ...

sa.save_sites_to_json(single_sites_lists, filename="single_sites_lists.json")


There are 30 unique sites out of 72.
Sites data saved to 'single_sites_lists.json' with None replaced by 'null'.


In [ ]:

# Load the trajectory file

# Load the trajectory file
unique_sites = read('unique_sites.traj', index=1)
view(unique_sites, viewer='x3d')

In [ ]:
#test
cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slab, surface, composition_effect=True)
single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())
sa.save_sites_to_json(single_sites_lists, filename="single_sites_lists_customsurf.json")
#
#traj = Trajectory("unique_sites.traj", "w")
#for g in single_geoms:
#    traj.write(g)
#traj.close()

In [ ]:
print(cas.get_sites())
len(cas.get_sites())

In [ ]:
[cas.get_sites()[i]['morphology'] for i in range(len(cas.get_sites()))]

In [ ]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [ ]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [ ]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [ ]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [ ]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


In [ ]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct 3-fold site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


In [ ]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


In [ ]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)
